<a href="https://colab.research.google.com/github/[USERNAME]/northstar-analytics-cw1/blob/main/notebooks/05_mongodb_optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05 — MongoDB Atlas: Indexing and Query Optimisation

**NorthStar Analytics — Coursework 1, Section 7**

Build indexes on the service_cases collection and verify performance improvement using `explain('executionStats')`.

In [ ]:
!pip install pymongo dnspython -q

from pymongo import MongoClient, ASCENDING, DESCENDING
from datetime import datetime
from google.colab import userdata

MONGO_URI = userdata.get('MONGO_URI')
client = MongoClient(MONGO_URI)
db = client['northstar']
cases = db['service_cases']

print(f'Cases in collection: {cases.count_documents({})}')
print(f'Existing indexes:')
for idx in cases.list_indexes(): print(f'  {idx["name"]}: {idx["key"]}')

## 1. Baseline — query without any custom index

In [ ]:
# Drop any existing indexes except _id
for idx in cases.list_indexes():
    if idx['name'] != '_id_':
        cases.drop_index(idx['name'])

# Baseline explain on the most operationally critical query
baseline = cases.find(
    {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'}
).explain('executionStats')

plan = baseline['queryPlanner']['winningPlan']
stats = baseline['executionStats']

print(f'Stage           : {plan["stage"]}')
print(f'totalDocsExamined: {stats["totalDocsExamined"]}')
print(f'totalDocsReturned: {stats["totalDocsReturned"]}')
print(f'executionMillis : {stats["executionTimeMillis"]}')

baseline_docs_examined = stats['totalDocsExamined']
baseline_ms = stats['executionTimeMillis']

Expected baseline: **COLLSCAN**, totalDocsExamined ≈ 886, totalDocsReturned ≈ 30.

## 2. Build the index plan

In [ ]:
# Index 1 — compound: zone + status (most operationally critical)
cases.create_index([('route.pickup_zone', ASCENDING),
                    ('delivery.status', ASCENDING)],
                   name='idx_zone_status', background=True)

# Index 2 — customer lookup
cases.create_index([('customer.customer_id', ASCENDING)],
                   name='idx_customer_id', background=True)

# Index 3 — service segment analytics
cases.create_index([('service_type', ASCENDING)],
                   name='idx_service_type', background=True)

# Index 4 — driver scorecards
cases.create_index([('delivery.driver_ref', ASCENDING)],
                   name='idx_driver_ref', background=True)

# Index 5 — multikey on incidents.severity (array)
cases.create_index([('incidents.severity', ASCENDING)],
                   name='idx_inc_severity', background=True)

# Index 6 — recent-cases dashboards
cases.create_index([('created_at', DESCENDING)],
                   name='idx_created_desc', background=True)

# Index 7 — covered query for rating dashboards
cases.create_index(
    [('route.pickup_zone', ASCENDING),
     ('delivery.status', ASCENDING),
     ('delivery.customer_rating', ASCENDING)],
    name='idx_zone_status_rating_cov', background=True)

print('Built indexes:')
for idx in cases.list_indexes():
    print(f'  {idx["name"]}: {dict(idx["key"])}')

## 3. Post-index measurement

In [ ]:
optimised = cases.find(
    {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'}
).explain('executionStats')

plan2 = optimised['queryPlanner']['winningPlan']
stats2 = optimised['executionStats']

# Walk inputStage to find IXSCAN node
def find_index_name(plan_node):
    if 'indexName' in plan_node:
        return plan_node['indexName']
    if 'inputStage' in plan_node:
        return find_index_name(plan_node['inputStage'])
    return None

index_used = find_index_name(plan2)
print(f'Stage            : {plan2["stage"]}')
print(f'Index used       : {index_used}')
print(f'totalDocsExamined: {stats2["totalDocsExamined"]}')
print(f'totalKeysExamined: {stats2["totalKeysExamined"]}')
print(f'totalDocsReturned: {stats2["totalDocsReturned"]}')
print(f'executionMillis  : {stats2["executionTimeMillis"]}')
print()
print(f'Improvement summary:')
print(f'  docs examined : {baseline_docs_examined} → {stats2["totalDocsExamined"]}'
      f' ({(1 - stats2["totalDocsExamined"]/baseline_docs_examined)*100:.1f}% reduction)')
print(f'  exec time (ms): {baseline_ms} → {stats2["executionTimeMillis"]}')

## 4. Covered query — fastest possible read pattern

When the index includes every field used in the filter and the projection, MongoDB returns the result entirely from the index without fetching any documents.

In [ ]:
covered = cases.find(
    {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'},
    {'delivery.customer_rating': 1, '_id': 0}
).explain('executionStats')

print(f'Stage chain:')
def walk(node, indent=0):
    print('  ' * indent + node['stage'] +
          (f' (index: {node.get("indexName")})' if node.get('indexName') else ''))
    if 'inputStage' in node:
        walk(node['inputStage'], indent + 1)
walk(covered['queryPlanner']['winningPlan'])
print(f'\ntotalDocsExamined: {covered["executionStats"]["totalDocsExamined"]}'
      f'  (0 means a covered query — answered from index only)')

## 5. Aggregation pipeline optimisation

Three patterns: $match early, $project to drop unused fields, $sort backed by an index.

In [ ]:
from datetime import datetime

optimised_pipeline = [
    {'$match': {'delivery.status': {'$in': ['Failed', 'Delayed']},
                'created_at': {'$gte': datetime(2025, 1, 1)}}},
    {'$project': {
        'route.pickup_zone': 1,
        'delivery.status': 1,
        'delivery.customer_rating': 1
    }},
    {'$group': {
        '_id': '$route.pickup_zone',
        'n_problem': {'$sum': 1},
        'avg_rating': {'$avg': '$delivery.customer_rating'}
    }},
    {'$sort': {'n_problem': -1}}
]

explain = db.command('aggregate', 'service_cases',
                     pipeline=optimised_pipeline,
                     explain=True)

for d in cases.aggregate(optimised_pipeline):
    print(d)

## 6. Index utilisation report

Per-index access statistics from MongoDB's $indexStats — useful for retiring indexes that are not earning their keep.

In [ ]:
for stat in cases.aggregate([{'$indexStats': {}}]):
    print(f"{stat['name']:35s}  ops: {stat['accesses']['ops']:6d}"
          f"  since: {stat['accesses']['since']}")